# Scheduled vs Unscheduled CFM and OT-CFM

This notebook compares four training configurations on two 1-D tutorial distributions:

| Label | Flow matcher | Schedule |
|---|---|---|
| **I-CFM** | `ConditionalFlowMatcher` | identity (τ = t) |
| **Sched I-CFM** | `ConditionalFlowMatcher` | `SigmoidSchedule(k=5)` |
| **OT-CFM** | `ExactOptimalTransportConditionalFlowMatcher` | identity (τ = t) |
| **Sched OT-CFM** | `ExactOptimalTransportConditionalFlowMatcher` | `SigmoidSchedule(k=5)` |

### What does the schedule do?

A schedule τ: [0,1] → [0,1] reparameterises the interpolant:

$$X_t^\tau = \tau(t)\,X_1 + (1-\tau(t))\,X_0, \qquad u_\tau = \dot\tau(t)\,(X_1 - X_0)$$

With **SigmoidSchedule** τ traverses slowly near the endpoints and quickly through the middle, making the spatial Lipschitz constant of the optimal velocity field roughly uniform in time.  
See Tsimpos, Ren, Zech & Marzouk, *"Optimal Scheduling for Conditional Flow Matching"* (2024).

### Distributions

| Experiment | Source p₀ | Target p₁ |
|---|---|---|
| **Gaussian mixture** | N(0, 1) | ½N(−2, 0.01) + ½N(+2, 0.01) |
| **Crossing flows** | ½N(−3, 0.04) + ½N(+3, 0.04) | ½N(−1, 0.04) + ½N(+1, 0.04) |

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from torchcfm.conditional_flow_matching import (
    ConditionalFlowMatcher,
    ExactOptimalTransportConditionalFlowMatcher,
)
from torchcfm.schedules import IdentitySchedule, SigmoidSchedule

torch.manual_seed(42)
np.random.seed(42)

SIGMA      = 0.01
BATCH_SIZE = 512
N_ITER     = 5_000
LR         = 1e-3
N_ODE      = 200    # Euler steps for ODE integration
N_TRAJ     = 1_000  # particles for trajectory plot
N_SHOW     = 200    # trajectories actually drawn
SMOOTH_W   = 150    # loss smoothing window

## Distributions

In [ ]:
def _bimodal(n, centers, std):
    comp  = torch.randint(0, 2, (n,))
    means = torch.where(comp == 0, torch.tensor(centers[0]), torch.tensor(centers[1]))
    return (means + std * torch.randn(n)).unsqueeze(1)

# ── Gaussian mixture ─────────────────────────────────────────────────────────
def gm_source(n): return torch.randn(n, 1)
def gm_target(n): return _bimodal(n, [-2.0, 2.0], std=0.1)

GM_TARGET_CENTERS = [-2.0, 2.0]

# ── Crossing flows ────────────────────────────────────────────────────────────
def cf_source(n): return _bimodal(n, [-3.0, 3.0], std=0.2)
def cf_target(n): return _bimodal(n, [-1.0, 1.0], std=0.2)

CF_TARGET_CENTERS = [-1.0, 1.0]

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

for ax, src_fn, tgt_fn, tgt_c, title in [
    (axes[0], gm_source, gm_target, GM_TARGET_CENTERS, 'Gaussian mixture'),
    (axes[1], cf_source, cf_target, CF_TARGET_CENTERS, 'Crossing flows'),
]:
    ax.hist(src_fn(4000).numpy(), bins=80, density=True, alpha=0.6, label='Source p₀')
    ax.hist(tgt_fn(4000).numpy(), bins=80, density=True, alpha=0.6, label='Target p₁')
    for c in tgt_c:
        ax.axvline(c, color='red', linestyle='--', linewidth=1, alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('density')
    ax.legend()

plt.tight_layout()
plt.show()

## Schedule visualisation

τ(t) and τ̇(t) for the two schedules used in this notebook.

In [ ]:
t_grid = torch.linspace(0, 1, 300)
sched_id  = IdentitySchedule()
sched_sig = SigmoidSchedule(k=5.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(t_grid, sched_id.tau(t_grid).detach(),   label='Identity  τ(t) = t',         lw=2)
ax.plot(t_grid, sched_sig.tau(t_grid).detach(),  label='Sigmoid k=5  τ(t)',           lw=2)
ax.plot([0,1], [0,1], 'k--', alpha=0.3, lw=1)
ax.set_xlabel('t')
ax.set_ylabel('τ(t)')
ax.set_title('Schedule map  τ(t)')
ax.legend()

ax = axes[1]
ax.plot(t_grid, sched_id.tau_dot(t_grid).detach(),   label='Identity  τ̇(t) = 1',  lw=2)
ax.plot(t_grid, sched_sig.tau_dot(t_grid).detach(),  label='Sigmoid k=5  τ̇(t)',   lw=2)
ax.axhline(1.0, color='k', linestyle='--', alpha=0.3, lw=1)
ax.set_xlabel('t')
ax.set_ylabel('τ̇(t)')
ax.set_title('Schedule derivative  τ̇(t)  — scales velocity targets')
ax.legend()

plt.tight_layout()
plt.show()
print('τ̇(0.5) for SigmoidSchedule(k=5):', sched_sig.tau_dot(torch.tensor([0.5])).item())

## Shared training infrastructure

In [ ]:
class MLP1D(nn.Module):
    """Small time-conditioned MLP for 1-D flow matching: input (x, t) → velocity."""

    def __init__(self, width=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, width), nn.SELU(),
            nn.Linear(width, width), nn.SELU(),
            nn.Linear(width, width), nn.SELU(),
            nn.Linear(width, 1),
        )

    def forward(self, xt, t):
        return self.net(torch.cat([xt, t.unsqueeze(-1)], dim=-1))


def train(fm, source_fn, target_fn, label='', seed=42):
    """Train MLP1D with the given flow matcher; return (model, loss_history)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MLP1D()
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    losses = []
    t0 = time.time()

    for k in range(N_ITER):
        opt.zero_grad()
        x0 = source_fn(BATCH_SIZE)
        x1 = target_fn(BATCH_SIZE)
        t, xt, ut = fm.sample_location_and_conditional_flow(x0, x1)
        vt   = model(xt, t)
        loss = torch.mean((vt - ut) ** 2)
        loss.backward()
        opt.step()
        losses.append(loss.item())

        if label and (k + 1) % 1000 == 0:
            print(f"{label} | iter {k+1:>5d} | loss {loss.item():.4f} | {time.time()-t0:.1f}s")

    return model, losses


@torch.no_grad()
def euler_integrate(model, x_init, steps=N_ODE):
    """Simple Euler ODE integration from t=0 to t=1; returns trajectory (steps+1, N, 1)."""
    dt  = 1.0 / steps
    x   = x_init.clone()
    traj = [x.clone()]
    for i in range(steps):
        t_val = torch.full((x.shape[0],), i * dt)
        vt  = model(x, t_val)
        x   = x + dt * vt
        traj.append(x.clone())
    return torch.stack(traj, dim=0)   # (steps+1, N, 1)


def smooth(arr, w=SMOOTH_W):
    return np.convolve(arr, np.ones(w) / w, mode='valid')

## Four flow matchers

We instantiate the four configurations once and reuse them for both experiments.

In [ ]:
CONFIGS = {
    'I-CFM':          ConditionalFlowMatcher(sigma=SIGMA),
    'Sched I-CFM':    ConditionalFlowMatcher(sigma=SIGMA, schedule=SigmoidSchedule(k=5.0)),
    'OT-CFM':         ExactOptimalTransportConditionalFlowMatcher(sigma=SIGMA),
    'Sched OT-CFM':   ExactOptimalTransportConditionalFlowMatcher(sigma=SIGMA, schedule=SigmoidSchedule(k=5.0)),
}

COLORS = {
    'I-CFM':        'steelblue',
    'Sched I-CFM':  'cornflowerblue',
    'OT-CFM':       'darkorange',
    'Sched OT-CFM': 'tomato',
}

LINESTYLES = {
    'I-CFM':        '-',
    'Sched I-CFM':  '--',
    'OT-CFM':       '-',
    'Sched OT-CFM': '--',
}

---
# Experiment 1 — Gaussian Mixture

Source **p₀ = N(0,1)**, Target **p₁ = ½N(−2, 0.01) + ½N(+2, 0.01)**

The sharp target modes (σ=0.1) make the velocity field hard to learn uniformly across time with the identity schedule.

In [ ]:
print('Training on Gaussian mixture ...')
gm_results = {}
for name, fm in CONFIGS.items():
    model, losses = train(fm, gm_source, gm_target, label=name)
    gm_results[name] = {'model': model, 'losses': losses}
print('Done.')

### Training loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, pairs, title in [
    (axes[0], ['I-CFM', 'Sched I-CFM'],  'Independent coupling (I-CFM)'),
    (axes[1], ['OT-CFM', 'Sched OT-CFM'], 'OT coupling (OT-CFM)'),
]:
    for name in pairs:
        s = smooth(gm_results[name]['losses'])
        ax.plot(s, color=COLORS[name], ls=LINESTYLES[name], label=name, lw=2)
    ax.set_xlabel('iteration')
    ax.set_ylabel('CFM loss (smoothed)')
    ax.set_title(f'Gaussian mixture — {title}')
    ax.legend()

plt.suptitle('Dashed = SigmoidSchedule(k=5)', y=0.01, fontsize=9)
plt.tight_layout()
plt.show()

print('Final losses:')
for name, r in gm_results.items():
    print(f'  {name:<18s}: {r["losses"][-1]:.5f}')

### Particle trajectories

In [ ]:
torch.manual_seed(0)
x_init_gm = gm_source(N_TRAJ)
gm_trajs = {name: euler_integrate(r['model'], x_init_gm) for name, r in gm_results.items()}
t_np = np.linspace(0, 1, N_ODE + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)

for ax, name in zip(axes.ravel(), CONFIGS):
    traj = gm_trajs[name].numpy()   # (T, N, 1)
    x_start = traj[0, :N_SHOW, 0]
    vmin, vmax = x_start.min(), x_start.max()
    for i in range(N_SHOW):
        c = plt.cm.coolwarm((x_start[i] - vmin) / (vmax - vmin + 1e-8))
        ax.plot(t_np, traj[:, i, 0], color=c, alpha=0.2, linewidth=0.6)
    ax.scatter(np.zeros(N_SHOW), traj[0,  :N_SHOW, 0], s=5, c='black', zorder=3)
    ax.scatter(np.ones(N_SHOW),  traj[-1, :N_SHOW, 0], s=5, c='blue',  zorder=3)
    for c_val in GM_TARGET_CENTERS:
        ax.axhline(c_val, color='red', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('time t')
    ax.set_ylabel('position x')

plt.suptitle('Gaussian mixture — particle trajectories  (red dashed = target mode centres)', fontsize=12)
plt.tight_layout()
plt.show()

### Generated samples vs true target

In [ ]:
x_tgt_gm = gm_target(4000).numpy()

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=True)

for ax, name in zip(axes.ravel(), CONFIGS):
    gen = gm_trajs[name][-1, :, 0].numpy()
    ax.hist(x_tgt_gm,         bins=80, density=True, alpha=0.55, color='darkorange', label='True p₁')
    ax.hist(gen[:, np.newaxis], bins=80, density=True, alpha=0.55,
            color=COLORS[name], label=f'{name} generated')
    for c_val in GM_TARGET_CENTERS:
        ax.axvline(c_val, color='red', linestyle='--', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('x')
    ax.legend(fontsize=8)

axes[0, 0].set_ylabel('density')
axes[1, 0].set_ylabel('density')
plt.suptitle('Gaussian mixture — generated vs true p₁', fontsize=12)
plt.tight_layout()
plt.show()

### Velocity field heatmaps

With the **identity schedule** the velocity target is constant `x₁ − x₀` for all t, so the field has a hard job near the sharp modes at t≈1.  
With the **sigmoid schedule** the field is large at mid-t (where τ̇ peaks) and small at the endpoints — the network sees a smoother learning signal throughout training.

In [ ]:
x_grid = np.linspace(-4, 4, 200)
t_grid_np = np.linspace(0, 1, 100)
XX, TT = np.meshgrid(x_grid, t_grid_np)
x_flat = torch.tensor(XX.ravel(), dtype=torch.float32).unsqueeze(1)
t_flat = torch.tensor(TT.ravel(), dtype=torch.float32)

@torch.no_grad()
def eval_field(model):
    return model(x_flat, t_flat).numpy().reshape(XX.shape)

vf_gm = {name: eval_field(r['model']) for name, r in gm_results.items()}
vabs  = max(np.abs(v).max() for v in vf_gm.values())

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True, sharex=True)

for ax, name in zip(axes.ravel(), CONFIGS):
    im = ax.pcolormesh(x_grid, t_grid_np, vf_gm[name],
                       cmap='RdBu_r', vmin=-vabs, vmax=vabs, shading='auto')
    plt.colorbar(im, ax=ax, label='vθ(t, x)')
    for c_val in GM_TARGET_CENTERS:
        ax.axvline(c_val, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('position x')
    ax.set_ylabel('time t')

plt.suptitle('Gaussian mixture — learned velocity fields  (blue=left, red=right  |  dashed = target modes)', fontsize=11)
plt.tight_layout()
plt.show()

### Velocity field slices at fixed t

Slices at t ∈ {0, 0.25, 0.5, 0.75, 1} show how the schedule shifts the "commitment" of the field across time.

In [ ]:
T_SLICES = [0.0, 0.25, 0.5, 0.75, 1.0]
slice_colors = plt.cm.viridis(np.linspace(0, 1, len(T_SLICES)))
x_line = torch.linspace(-4, 4, 400).unsqueeze(1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True, sharex=True)

for ax, name in zip(axes.ravel(), CONFIGS):
    model = gm_results[name]['model']
    model.eval()
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    for t_val, col in zip(T_SLICES, slice_colors):
        t_line = torch.full((400,), t_val)
        with torch.no_grad():
            v = model(x_line, t_line).numpy().ravel()
        ax.plot(x_line.numpy().ravel(), v, color=col, label=f't={t_val}')
    for c_val in GM_TARGET_CENTERS:
        ax.axvline(c_val, color='red', ls='--', lw=1, alpha=0.5)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('x')
    ax.set_ylabel('vθ(t, x)')
    ax.legend(fontsize=7)

plt.suptitle('Gaussian mixture — velocity slices at fixed t  (red dashed = target mode centres)', fontsize=11)
plt.tight_layout()
plt.show()

---
# Experiment 2 — Crossing Flows

Source **p₀ = ½N(−3, 0.04) + ½N(+3, 0.04)**, Target **p₁ = ½N(−1, 0.04) + ½N(+1, 0.04)**

With **I-CFM** a left particle is equally likely to be paired with either target mode, producing crossing trajectories and a near-zero "dead zone" in the velocity field at x≈0. **OT-CFM** eliminates crossing by using the monotone pairing. The **schedule** further reduces variance in the velocity targets across time.

In [ ]:
print('Training on crossing flows ...')
cf_results = {}
for name, fm in CONFIGS.items():
    model, losses = train(fm, cf_source, cf_target, label=name)
    cf_results[name] = {'model': model, 'losses': losses}
print('Done.')

### Training loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, pairs, title in [
    (axes[0], ['I-CFM', 'Sched I-CFM'],   'Independent coupling (I-CFM)'),
    (axes[1], ['OT-CFM', 'Sched OT-CFM'], 'OT coupling (OT-CFM)'),
]:
    for name in pairs:
        s = smooth(cf_results[name]['losses'])
        ax.plot(s, color=COLORS[name], ls=LINESTYLES[name], label=name, lw=2)
    ax.set_xlabel('iteration')
    ax.set_ylabel('CFM loss (smoothed)')
    ax.set_title(f'Crossing flows — {title}')
    ax.legend()

plt.suptitle('Dashed = SigmoidSchedule(k=5)', y=0.01, fontsize=9)
plt.tight_layout()
plt.show()

print('Final losses:')
for name, r in cf_results.items():
    print(f'  {name:<18s}: {r["losses"][-1]:.5f}')

### Particle trajectories

In [ ]:
torch.manual_seed(0)
x_init_cf = cf_source(N_TRAJ)
cf_trajs = {name: euler_integrate(r['model'], x_init_cf) for name, r in cf_results.items()}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)

for ax, name in zip(axes.ravel(), CONFIGS):
    traj = cf_trajs[name].numpy()
    x_start = traj[0, :N_SHOW, 0]
    vmin, vmax = x_start.min(), x_start.max()
    for i in range(N_SHOW):
        c = plt.cm.coolwarm((x_start[i] - vmin) / (vmax - vmin + 1e-8))
        ax.plot(t_np, traj[:, i, 0], color=c, alpha=0.2, linewidth=0.6)
    ax.scatter(np.zeros(N_SHOW), traj[0,  :N_SHOW, 0], s=5, c='black', zorder=3)
    ax.scatter(np.ones(N_SHOW),  traj[-1, :N_SHOW, 0], s=5, c='blue',  zorder=3)
    for c_val in CF_TARGET_CENTERS:
        ax.axhline(c_val, color='red', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('time t')
    ax.set_ylabel('position x')

plt.suptitle(
    'Crossing flows — particle trajectories\n'
    'I-CFM: crossing expected  |  OT-CFM: monotone  |  red dashed = target mode centres',
    fontsize=11,
)
plt.tight_layout()
plt.show()

### Generated samples vs true target

In [ ]:
x_tgt_cf = cf_target(4000).numpy()

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=True)

for ax, name in zip(axes.ravel(), CONFIGS):
    gen = cf_trajs[name][-1, :, 0].numpy()
    ax.hist(x_tgt_cf,          bins=80, density=True, alpha=0.55, color='darkorange', label='True p₁')
    ax.hist(gen[:, np.newaxis],  bins=80, density=True, alpha=0.55,
            color=COLORS[name], label=f'{name} generated')
    for c_val in CF_TARGET_CENTERS:
        ax.axvline(c_val, color='red', linestyle='--', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('x')
    ax.legend(fontsize=8)

axes[0, 0].set_ylabel('density')
axes[1, 0].set_ylabel('density')
plt.suptitle('Crossing flows — generated vs true p₁', fontsize=12)
plt.tight_layout()
plt.show()

### Velocity field heatmaps

**I-CFM**: crossing opposite flows average to ~0 near x=0, creating a dead zone.  
**OT-CFM**: clean antisymmetric field, decisive at all t.  
**Scheduled variants**: the colour intensity shifts toward mid-t where τ̇ peaks.

In [ ]:
x_grid_cf = np.linspace(-4.5, 4.5, 250)
t_grid_cf = np.linspace(0, 1, 120)
XX_cf, TT_cf = np.meshgrid(x_grid_cf, t_grid_cf)
x_flat_cf = torch.tensor(XX_cf.ravel(), dtype=torch.float32).unsqueeze(1)
t_flat_cf = torch.tensor(TT_cf.ravel(), dtype=torch.float32)

@torch.no_grad()
def eval_field_cf(model):
    return model(x_flat_cf, t_flat_cf).numpy().reshape(XX_cf.shape)

vf_cf = {name: eval_field_cf(r['model']) for name, r in cf_results.items()}
vabs_cf = max(np.abs(v).max() for v in vf_cf.values())

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True, sharex=True)

for ax, name in zip(axes.ravel(), CONFIGS):
    im = ax.pcolormesh(x_grid_cf, t_grid_cf, vf_cf[name],
                       cmap='RdBu_r', vmin=-vabs_cf, vmax=vabs_cf, shading='auto')
    plt.colorbar(im, ax=ax, label='vθ(t, x)')
    # overlay a few trajectories for context
    traj = cf_trajs[name].numpy()
    for i in range(0, N_SHOW, 4):
        ax.plot(traj[:, i, 0], t_np, color='white', alpha=0.3, linewidth=0.6)
    for c_val in CF_TARGET_CENTERS:
        ax.axvline(c_val, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('position x')
    ax.set_ylabel('time t')

plt.suptitle(
    'Crossing flows — learned velocity fields  '
    '(blue=left, red=right  |  white = particle trajectories  |  dashed = target modes)',
    fontsize=10,
)
plt.tight_layout()
plt.show()

### Velocity field slices — crossing flows

In [ ]:
x_line_cf = torch.linspace(-4.5, 4.5, 500).unsqueeze(1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True, sharex=True)

for ax, name in zip(axes.ravel(), CONFIGS):
    model = cf_results[name]['model']
    model.eval()
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    for t_val, col in zip(T_SLICES, slice_colors):
        t_line = torch.full((500,), t_val)
        with torch.no_grad():
            v = model(x_line_cf, t_line).numpy().ravel()
        ax.plot(x_line_cf.numpy().ravel(), v, color=col, label=f't={t_val}', lw=1.5)
    for c_val in CF_TARGET_CENTERS:
        ax.axvline(c_val, color='red', ls='--', lw=1, alpha=0.5)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('x')
    ax.set_ylabel('vθ(t, x)')
    ax.legend(fontsize=7)

plt.suptitle(
    'Crossing flows — velocity slices at fixed t\n'
    'I-CFM: dead zone at x≈0  |  OT-CFM: clean antisymmetry',
    fontsize=11,
)
plt.tight_layout()
plt.show()

---
## Summary

| | I-CFM | Sched I-CFM | OT-CFM | Sched OT-CFM |
|---|---|---|---|---|
| **Coupling** | independent | independent | OT (monotone) | OT (monotone) |
| **Schedule** | τ = t | SigmoidSchedule(k=5) | τ = t | SigmoidSchedule(k=5) |
| **Velocity target** | x₁ − x₀ | τ̇(t)·(x₁ − x₀) | x₁ − x₀ | τ̇(t)·(x₁ − x₀) |
| **Crossing flows** | crossing | crossing | monotone | monotone |

**Key observations:**

- **OT coupling** (columns 3–4) eliminates crossing trajectories and achieves lower loss regardless of schedule — this is the dominant effect on the crossing-flows distribution.
- **Scheduling** (columns 2 and 4) rescales the velocity targets by τ̇(t): small near the endpoints, large at mid-t. The network sees a smoother, more uniform training signal across time. On the sharper Gaussian-mixture target this can reduce the final loss.
- **Scheduled OT-CFM** combines both benefits and is the method recommended by Tsimpos et al. (2024) for best approximation error per parameter.